# QADD v4.2.0 — Reviewed cohort extraction and empirical validation

**Family:** Additive interference  
**Measurement target:** recorded pause-region energy and selected additive-source structure  
**Input contract:** frozen `primary_speech / primary`, `strict_speech / primary`, and `strict_internal_nonspeech / primary` intervals. The strict speech and strict pause views are already guarded; no second guard is applied.

This notebook extends the completed G1–G6 preflight. It performs the corrected 519-recording cohort extraction, count-matched hum-null calibration, support and robustness audits, empirical characterization, repeated-recording persistence, participant-balanced summaries, ML-facing export, and Panels D–H/J.

It **cannot freeze QADD**. Feature-specific G10 decisions are made only after the executed cohort outputs are reviewed.


In [ ]:
from __future__ import annotations

from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import os
import shutil
import subprocess
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown


def find_project_root() -> Path:
    override = os.environ.get("PAPER1_PROJECT_ROOT", "").strip()
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
        raise FileNotFoundError(f"PAPER1_PROJECT_ROOT is invalid: {candidate}")
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Open this notebook from inside the Paper 1 repository, "
        "or set PAPER1_PROJECT_ROOT."
    )


ROOT = find_project_root()
REVIEWED_SRC = ROOT / "src"
ORIGINAL_SRC = ROOT / "src"
for path in [REVIEWED_SRC, ORIGINAL_SRC]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from paper1_qc.media import decode_audio_views
from paper1_qc_reviewed.qadd_v420 import (
    ANALYSIS_FEATURES,
    DEFAULT_PARAMETERS,
    MEASUREMENT_VERSION,
    TimeInterval,
    apply_hum_null_calibration,
    cluster_delete_one_diagnostics,
    compare_reconstruction,
    erode_intervals,
    extract_qadd,
    feature_registry_frame,
    summarize_cluster_deletion,
)
from paper1_qc_reviewed.qadd_v420_cohort import (
    CANONICAL_PAUSE_VIEW,
    CANONICAL_PRIMARY_VIEW,
    CANONICAL_PROFILE,
    CANONICAL_SPEECH_VIEW,
    COHORT_ORCHESTRATION_VERSION,
    as_bool,
    attach_interval_provenance,
    canonical_interval_contract,
    conservative_nonincreasing_thresholds,
    observed_hum_support_grid,
    empirical_feature_summary,
    hash_inventory,
    hum_null_calibration_grid,
    hum_null_window_pool,
    intervals_for_recording,
    json_safe,
    measurement_long_frame,
    model_interface_frame,
    participant_balanced_resampling,
    participant_balanced_summary,
    repeated_recording_persistence,
    resolve_media_path,
    select_hum_null_reference,
    sha256_file,
    write_json,
)

# ---------------------------- execution controls ----------------------------
RUN_PACKAGE_TESTS = True
RUN_COHORT_EXTRACTION = False  # installer changes this to True in the local run copy
VERIFY_MEDIA_HASHES = True
REBUILD_CHECKPOINTS = False
RUN_COHORT_ROBUSTNESS = True
BUILD_GALLERY = True

HUM_NULL_POOL_SIZE = 4000
HUM_NULL_ITERATIONS = 3000
MAX_BOUNDARY_RECORDINGS = 120
PARTICIPANT_BALANCED_ITERATIONS = 1000
GALLERY_RECORDING_LIMIT = 12

MEDIA_ROOT_OVERRIDE = None  # installer inserts the local Bamboo_passage_only path
MEDIA_PATH_MAP = {}

PUBLISH_AND_FREEZE = False
SCIENTIFIC_REVIEW_DECISION = "PENDING"
SCIENTIFIC_REVIEWER = ""
SCIENTIFIC_REVIEW_RATIONALE = ""

PARAMETERS = DEFAULT_PARAMETERS
FS = PARAMETERS.analysis_sample_rate_hz

LEGACY_MAIN = ROOT / "MAIN outputs"
MAIN_REVIEWED = ROOT / "MAIN outputs/02_FEATURE_REVIEWED"
STAGE = ROOT / "MAIN outputs/02_FEATURE_REVIEWED/00_working_candidates" / "additive_interference" / MEASUREMENT_VERSION
TABLES = STAGE / "tables"
LEDGERS = STAGE / "ledgers"
VALIDATION = STAGE / "validation"
FIGURES = STAGE / "figures"
GALLERIES = STAGE / "galleries"
AUDIT = STAGE / "audit"
MANIFESTS = STAGE / "manifests"
CHECKPOINTS = STAGE / "checkpoints"
for directory in [
    TABLES, LEDGERS, VALIDATION, FIGURES, GALLERIES,
    AUDIT, MANIFESTS, CHECKPOINTS
]:
    directory.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

print("Project root:", ROOT)
print("Measurement:", MEASUREMENT_VERSION)
print("Orchestration:", COHORT_ORCHESTRATION_VERSION)
print("Run cohort:", RUN_COHORT_EXTRACTION)
print("Stage:", STAGE)


In [ ]:
def save_table(frame: pd.DataFrame, path_without_suffix: Path, *, parquet: bool = True) -> dict:
    path_without_suffix = Path(path_without_suffix)
    path_without_suffix.parent.mkdir(parents=True, exist_ok=True)
    csv_path = path_without_suffix.with_suffix(".csv")
    frame.to_csv(csv_path, index=False)
    result = {"csv": str(csv_path), "csv_sha256": sha256_file(csv_path)}
    if parquet:
        parquet_path = path_without_suffix.with_suffix(".parquet")
        try:
            frame.to_parquet(parquet_path, index=False)
            result.update(
                {"parquet": str(parquet_path), "parquet_sha256": sha256_file(parquet_path)}
            )
        except Exception as exc:
            result["parquet_error"] = f"{type(exc).__name__}: {exc}"
    return result


def save_figure_bundle(
    figure,
    *,
    stem: str,
    source_data: pd.DataFrame,
    caption: str,
    provenance: dict,
) -> dict:
    FIGURES.mkdir(parents=True, exist_ok=True)
    png = FIGURES / f"{stem}.png"
    svg = FIGURES / f"{stem}.svg"
    pdf = FIGURES / f"{stem}.pdf"
    source = FIGURES / f"{stem}.source.csv"
    caption_path = FIGURES / f"{stem}.caption.md"
    provenance_path = FIGURES / f"{stem}.provenance.json"
    figure.savefig(png, dpi=300, bbox_inches="tight")
    figure.savefig(svg, bbox_inches="tight")
    figure.savefig(pdf, bbox_inches="tight")
    source_data.to_csv(source, index=False)
    caption_path.write_text(caption.strip() + "\n", encoding="utf-8")
    write_json(
        {
            **provenance,
            "measurement_version": MEASUREMENT_VERSION,
            "source_csv_sha256": sha256_file(source),
            "feature_values_recomputed_by_figure": False,
        },
        provenance_path,
    )
    return {
        "panel_stem": stem,
        "png": str(png),
        "svg": str(svg),
        "pdf": str(pdf),
        "source_csv": str(source),
        "caption": str(caption_path),
        "provenance": str(provenance_path),
    }


def write_parquet_part(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(path, index=False)


def read_record_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def finite_iqr(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors="coerce")
    values = values[np.isfinite(values)]
    return float(values.quantile(0.75) - values.quantile(0.25)) if len(values) else np.nan


def status_is_available(value) -> bool:
    return str(value).startswith("ok_")


def subject_column_for(frame: pd.DataFrame) -> str:
    for column in ["SubjectID", "subject_id", "subject", "ID_norm"]:
        if column in frame and frame[column].notna().any():
            return column
    raise ValueError("No subject identifier column was found")


def date_column_for(frame: pd.DataFrame) -> str:
    for column in [
        "recording_date_analysis", "Recording date", "date_parsed", "recording_date"
    ]:
        if column in frame and frame[column].notna().any():
            return column
    raise ValueError("No recording-date column was found")


def select_evenly(frame: pd.DataFrame, count: int, *, sort_columns: list[str]) -> pd.DataFrame:
    if len(frame) <= count:
        return frame.copy()
    ordered = frame.sort_values(sort_columns).reset_index(drop=True)
    positions = np.linspace(0, len(ordered) - 1, count).round().astype(int)
    return ordered.iloc[np.unique(positions)].copy()


def ledger_checkpoint_paths(recording_id: str) -> dict[str, Path]:
    safe = str(recording_id).replace("/", "_").replace("\\", "_")
    return {
        "record": CHECKPOINTS / "records" / f"{safe}.json",
        "frame": CHECKPOINTS / "frame_parts" / f"{safe}.parquet",
        "interval": CHECKPOINTS / "interval_parts" / f"{safe}.parquet",
        "spectral": CHECKPOINTS / "spectral_parts" / f"{safe}.parquet",
        "reconstruction": CHECKPOINTS / "reconstruction_parts" / f"{safe}.csv",
        "deletion": CHECKPOINTS / "deletion_parts" / f"{safe}.parquet",
        "error": CHECKPOINTS / "errors" / f"{safe}.json",
    }


def checkpoint_complete(paths: dict[str, Path]) -> bool:
    return all(paths[key].exists() for key in ["record", "frame", "interval", "spectral", "reconstruction"])


def media_hash_column(frame: pd.DataFrame) -> str | None:
    for column in ["media_sha256", "selected_media_sha256", "sha256"]:
        if column in frame:
            return column
    return None


In [ ]:
preflight_manifest_path = MANIFESTS / "qadd_v420_preflight_manifest.json"
if not preflight_manifest_path.exists():
    raise FileNotFoundError(
        "The completed QADD v4.2.0 preflight manifest is missing. "
        "Run the reviewed preflight notebook first."
    )
preflight_manifest = json.loads(preflight_manifest_path.read_text(encoding="utf-8"))
preflight_checks = pd.read_csv(TABLES / "qadd_v420_preflight_all_checks.csv")
preflight_gates = pd.read_csv(TABLES / "qadd_v420_gate_summary.csv")
preflight_figures = []
for stem in ["A_construct_response", "B_hum_discriminant_specificity", "C_transformation_contract"]:
    expected = [
        FIGURES / f"{stem}.png",
        FIGURES / f"{stem}.svg",
        FIGURES / f"{stem}.pdf",
        FIGURES / f"{stem}.source.csv",
        FIGURES / f"{stem}.caption.md",
        FIGURES / f"{stem}.provenance.json",
    ]
    preflight_figures.append(
        {
            "panel": stem[0],
            "stem": stem,
            "all_artifacts_exist": all(path.exists() for path in expected),
            "missing": ";".join(str(path.name) for path in expected if not path.exists()),
        }
    )
preflight_figure_check = pd.DataFrame(preflight_figures)
preflight_ok = bool(
    preflight_manifest.get("preflight_blocking_checks_pass", False)
    and preflight_checks["passed"].astype(bool).all()
    and preflight_figure_check["all_artifacts_exist"].all()
)
if not preflight_ok:
    raise RuntimeError("QADD preflight evidence is incomplete; cohort extraction is blocked.")

test_output = "not_run"
if RUN_PACKAGE_TESTS:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REVIEWED_SRC) + os.pathsep + env.get("PYTHONPATH", "")
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            str(ROOT / "tests" / "test_qadd_v420.py"),
            str(ROOT / "tests" / "test_qadd_v420_cohort.py"),
            "-q",
        ],
        cwd=ROOT,
        env=env,
        text=True,
        capture_output=True,
    )
    test_output = result.stdout + result.stderr
    print(test_output)
    if result.returncode != 0:
        raise RuntimeError("Reviewed QADD package tests failed")

save_table(preflight_figure_check, VALIDATION / "qadd_v420_preflight_figure_contract")
display(preflight_gates)
display(preflight_figure_check)
print("Preflight accepted for cohort execution.")


In [ ]:
DATA_FREEZE = LEGACY_MAIN / "00_DATA_FREEZE" / "v1"
SEG_FREEZE = LEGACY_MAIN / "01_SEGMENTATION_FREEZE" / "v1"
recordings_path = DATA_FREEZE / "frozen_bamboo_recordings.csv"
decisions_path = SEG_FREEZE / "frozen_segmentation_decisions.csv"
intervals_path = SEG_FREEZE / "frozen_segmentation_intervals.csv"

input_artifacts = pd.DataFrame(
    [
        {
            "artifact": "frozen recordings",
            "path": str(recordings_path),
            "exists": recordings_path.exists(),
            "sha256": sha256_file(recordings_path) if recordings_path.exists() else None,
        },
        {
            "artifact": "frozen segmentation decisions",
            "path": str(decisions_path),
            "exists": decisions_path.exists(),
            "sha256": sha256_file(decisions_path) if decisions_path.exists() else None,
        },
        {
            "artifact": "frozen segmentation intervals",
            "path": str(intervals_path),
            "exists": intervals_path.exists(),
            "sha256": sha256_file(intervals_path) if intervals_path.exists() else None,
        },
    ]
)
if not input_artifacts["exists"].all():
    raise FileNotFoundError(
        "One or more frozen input artifacts are missing:\n"
        + input_artifacts.to_string(index=False)
    )

recordings = pd.read_csv(recordings_path)
decisions = pd.read_csv(decisions_path)
intervals = pd.read_csv(intervals_path)
canonical_tables, canonical_summary = canonical_interval_contract(decisions, intervals)

eligible_ids = (
    decisions.loc[
        as_bool(decisions["segmentation_analysis_eligible"]),
        "logical_recording_id",
    ]
    .astype(str)
    .drop_duplicates()
)
if eligible_ids.duplicated().any():
    raise ValueError("Eligible recording identities are duplicated")

recordings["logical_recording_id"] = recordings["logical_recording_id"].astype(str)
frozen = recordings.loc[
    recordings["logical_recording_id"].isin(set(eligible_ids))
].copy()
if "media_freeze_eligible" in frozen:
    frozen = frozen.loc[as_bool(frozen["media_freeze_eligible"])]
if "analysis_eligible" in frozen:
    frozen = frozen.loc[as_bool(frozen["analysis_eligible"])]

if frozen["logical_recording_id"].duplicated().any():
    raise ValueError("Frozen recording table contains duplicate eligible IDs")
if set(frozen["logical_recording_id"]) != set(eligible_ids):
    raise ValueError(
        f"Frozen recording/decision ID mismatch: recordings={len(frozen)}, "
        f"decisions={len(eligible_ids)}"
    )

subject_column = subject_column_for(frozen)
date_column = date_column_for(frozen)
hash_column = media_hash_column(frozen)

g1_input_checks = pd.DataFrame(
    [
        {
            "gate": "G1",
            "check": "all frozen input artifacts exist and are SHA-256 identified",
            "passed": bool(input_artifacts["exists"].all() and input_artifacts["sha256"].notna().all()),
            "observed": len(input_artifacts),
        },
        {
            "gate": "G1",
            "check": "eligible recording identities match the frozen media cohort",
            "passed": set(frozen["logical_recording_id"]) == set(eligible_ids),
            "observed": len(frozen),
        },
        {
            "gate": "G1",
            "check": "primary and strict speech are available for every eligible recording",
            "passed": bool(
                canonical_summary.loc[
                    canonical_summary["view"].isin(
                        [CANONICAL_PRIMARY_VIEW, CANONICAL_SPEECH_VIEW]
                    ),
                    "contract_pass",
                ].all()
            ),
            "observed": canonical_summary.to_json(orient="records"),
        },
        {
            "gate": "G1",
            "check": "strict pause absence is represented rather than silently synthesized",
            "passed": bool(
                canonical_summary.loc[
                    canonical_summary["view"].eq(CANONICAL_PAUSE_VIEW),
                    "contract_pass",
                ].all()
            ),
            "observed": int(
                canonical_summary.loc[
                    canonical_summary["view"].eq(CANONICAL_PAUSE_VIEW),
                    "missing_eligible_recording_count",
                ].iloc[0]
            ),
        },
        {
            "gate": "G1",
            "check": "QADD uses already-guarded strict speech and strict pauses without a second guard",
            "passed": True,
            "observed": "speech_intervals_are_guarded=True; pause_intervals_are_guarded=True",
        },
    ]
)
save_table(input_artifacts, VALIDATION / "qadd_v420_g1_input_artifacts")
save_table(canonical_summary, VALIDATION / "qadd_v420_canonical_interval_contract")
save_table(g1_input_checks, VALIDATION / "qadd_v420_g1_cohort_input_checks")

display(input_artifacts)
display(canonical_summary)
display(g1_input_checks)
print(f"Frozen cohort: {len(frozen)} recordings; subject column={subject_column}; date column={date_column}")


In [ ]:
record_rows = []
error_rows = []
start_time = time.time()

if RUN_COHORT_EXTRACTION:
    ffmpeg = shutil.which("ffmpeg")
    ffprobe = shutil.which("ffprobe")
    if not ffmpeg or not ffprobe:
        raise RuntimeError("ffmpeg and ffprobe are required for cohort extraction")

    frozen_by_id = frozen.set_index("logical_recording_id", drop=False)
    ordered_ids = frozen.sort_values("logical_recording_id")["logical_recording_id"].astype(str).tolist()

    for position, recording_id in enumerate(ordered_ids, start=1):
        paths = ledger_checkpoint_paths(recording_id)
        for path in paths.values():
            path.parent.mkdir(parents=True, exist_ok=True)

        if checkpoint_complete(paths) and not REBUILD_CHECKPOINTS:
            record_rows.append(read_record_json(paths["record"]))
            continue

        try:
            source_row = frozen_by_id.loc[recording_id]
            media_path = resolve_media_path(
                source_row["media_path"],
                media_root_override=(
                    Path(MEDIA_ROOT_OVERRIDE) if MEDIA_ROOT_OVERRIDE is not None else None
                ),
                media_path_map=MEDIA_PATH_MAP,
            )
            observed_hash = sha256_file(media_path) if VERIFY_MEDIA_HASHES else None
            expected_hash = (
                str(source_row[hash_column])
                if VERIFY_MEDIA_HASHES
                and hash_column is not None
                and pd.notna(source_row[hash_column])
                else None
            )
            if expected_hash and observed_hash.lower() != expected_hash.lower():
                raise ValueError(
                    f"Media SHA-256 mismatch for {recording_id}: "
                    f"expected={expected_hash}, observed={observed_hash}"
                )

            views = decode_audio_views(
                media_path,
                ffmpeg=ffmpeg,
                ffprobe=ffprobe,
                analysis_rate=FS,
            )
            primary, primary_meta = intervals_for_recording(
                canonical_tables[CANONICAL_PRIMARY_VIEW], recording_id
            )
            strict_speech, speech_meta = intervals_for_recording(
                canonical_tables[CANONICAL_SPEECH_VIEW], recording_id
            )
            strict_pause, pause_meta = intervals_for_recording(
                canonical_tables[CANONICAL_PAUSE_VIEW], recording_id
            )

            extraction = extract_qadd(
                views.analysis_16k,
                FS,
                primary_speech=primary,
                strict_speech=strict_speech,
                strict_internal_nonspeech=strict_pause,
                logical_recording_id=recording_id,
                speech_intervals_are_guarded=True,
                pause_intervals_are_guarded=True,
                parameters=PARAMETERS,
            )

            frames = extraction.frame_ledger
            pause_frames = attach_interval_provenance(
                frames.loc[frames["region"].eq("pause")],
                region="pause",
                canonical_intervals=pause_meta,
                canonical_view=CANONICAL_PAUSE_VIEW,
            )
            speech_frames = attach_interval_provenance(
                frames.loc[frames["region"].eq("speech")],
                region="speech",
                canonical_intervals=speech_meta,
                canonical_view=CANONICAL_SPEECH_VIEW,
            )
            frames = pd.concat([pause_frames, speech_frames], ignore_index=True)
            spectral = attach_interval_provenance(
                extraction.spectral_ledger,
                region="pause",
                canonical_intervals=pause_meta,
                canonical_view=CANONICAL_PAUSE_VIEW,
            )
            interval_ledger = attach_interval_provenance(
                extraction.interval_ledger,
                region="pause",
                canonical_intervals=pause_meta,
                canonical_view=CANONICAL_PAUSE_VIEW,
            )

            reconstruction = compare_reconstruction(extraction)
            reconstruction.insert(0, "logical_recording_id", recording_id)
            if not reconstruction["pass"].all():
                raise RuntimeError("Ledger reconstruction failed")

            deletion = cluster_delete_one_diagnostics(
                extraction.frame_ledger, extraction.spectral_ledger
            )

            record = {
                **extraction.recording,
                "qadd_primary_view": CANONICAL_PRIMARY_VIEW,
                "qadd_primary_profile": CANONICAL_PROFILE,
                "qadd_strict_speech_view": CANONICAL_SPEECH_VIEW,
                "qadd_strict_speech_profile": CANONICAL_PROFILE,
                "qadd_strict_pause_view": CANONICAL_PAUSE_VIEW,
                "qadd_strict_pause_profile": CANONICAL_PROFILE,
                "qadd_primary_interval_count_frozen": len(primary_meta),
                "qadd_strict_speech_interval_count_frozen": len(speech_meta),
                "qadd_strict_pause_interval_count_frozen": len(pause_meta),
                "file_name": source_row.get(
                    "selected_media_file_name",
                    source_row.get("file_name", media_path.name),
                ),
                "media_path_resolved": str(media_path),
                "media_sha256_verified": bool(
                    not VERIFY_MEDIA_HASHES or expected_hash is None or observed_hash == expected_hash
                ),
                "media_sha256_observed": observed_hash,
                "native_sample_rate_hz": int(views.sample_rate_native),
                "native_channels": int(views.native.shape[1]),
                "codec_name": views.probe.get("codec_name"),
                subject_column: source_row.get(subject_column),
                date_column: source_row.get(date_column),
            }
            for optional in [
                "SubjectID", "diagnosis_analysis", "Diagnosis", "recording_date_analysis",
                "Recording date", "date_parsed", "severity_bin"
            ]:
                if optional in source_row.index:
                    record[optional] = source_row.get(optional)

            write_json(record, paths["record"])
            write_parquet_part(frames, paths["frame"])
            write_parquet_part(interval_ledger, paths["interval"])
            write_parquet_part(spectral, paths["spectral"])
            reconstruction.to_csv(paths["reconstruction"], index=False)
            if len(deletion):
                write_parquet_part(deletion, paths["deletion"])
            elif paths["deletion"].exists():
                paths["deletion"].unlink()
            if paths["error"].exists():
                paths["error"].unlink()

            record_rows.append(json_safe(record))
        except Exception as exc:
            payload = {
                "logical_recording_id": recording_id,
                "error_type": type(exc).__name__,
                "message": str(exc),
            }
            write_json(payload, paths["error"])
            error_rows.append(payload)

        if position % 10 == 0 or position == len(ordered_ids):
            elapsed = time.time() - start_time
            print(
                f"[{position:03d}/{len(ordered_ids)}] "
                f"records={len(record_rows)} errors={len(error_rows)} "
                f"elapsed={elapsed/60:.1f} min"
            )
else:
    print("RUN_COHORT_EXTRACTION=False — no audio was processed.")

if RUN_COHORT_EXTRACTION:
    # Rebuild the recording table from all successful checkpoints, including
    # checkpoints reused from an interrupted prior run.
    record_rows = []
    error_rows = []
    for recording_id in frozen["logical_recording_id"].astype(str):
        paths = ledger_checkpoint_paths(recording_id)
        if paths["record"].exists():
            record_rows.append(read_record_json(paths["record"]))
        elif paths["error"].exists():
            error_rows.append(read_record_json(paths["error"]))
        else:
            error_rows.append(
                {
                    "logical_recording_id": recording_id,
                    "error_type": "MissingCheckpoint",
                    "message": "No successful or failed checkpoint exists",
                }
            )

recording_table = pd.DataFrame(record_rows)
extraction_errors = pd.DataFrame(
    error_rows, columns=["logical_recording_id", "error_type", "message"]
)
if len(recording_table):
    recording_table["logical_recording_id"] = recording_table[
        "logical_recording_id"
    ].astype(str)
    recording_table = recording_table.sort_values("logical_recording_id").reset_index(drop=True)

save_table(extraction_errors, AUDIT / "qadd_v420_extraction_errors", parquet=False)
print(f"Successful recordings: {len(recording_table)}")
print(f"Extraction errors: {len(extraction_errors)}")
display(extraction_errors.head(20))


In [ ]:
if RUN_COHORT_EXTRACTION and len(recording_table):
    observed_hum_window_counts = (
        pd.to_numeric(
            recording_table["qadd_hum_valid_window_count"], errors="coerce"
        )
        .dropna()
        .astype(int)
        .tolist()
    )
    support_grid = observed_hum_support_grid(observed_hum_window_counts)
    pool = hum_null_window_pool(
        parameters=PARAMETERS,
        pool_size=HUM_NULL_POOL_SIZE,
        seed=PARAMETERS.random_seed + 101,
    )
    hum_null_grid = hum_null_calibration_grid(
        pool,
        support_counts=support_grid,
        iterations=HUM_NULL_ITERATIONS,
        seed=PARAMETERS.random_seed + 102,
    )
    hum_null_grid = hum_null_grid.sort_values("window_count").reset_index(drop=True)
    hum_null_grid["null_p95_db_raw"] = hum_null_grid["null_p95_db"]
    # Enforce the theoretically non-increasing support relationship without
    # lowering any simulated P95 threshold.  Reverse cumulative maxima are
    # conservative; a cumulative minimum would be anti-conservative.
    raw_p95 = hum_null_grid["null_p95_db_raw"].to_numpy(float)
    hum_null_grid["null_p95_db"] = conservative_nonincreasing_thresholds(
        raw_p95
    )
    hum_null_grid["monotonic_adjustment_db"] = (
        hum_null_grid["null_p95_db"] - hum_null_grid["null_p95_db_raw"]
    )

    calibrated_rows = []
    for row in recording_table.to_dict(orient="records"):
        count = int(row.get("qadd_hum_valid_window_count") or 0)
        reference_count, threshold = select_hum_null_reference(count, hum_null_grid)
        calibrated = apply_hum_null_calibration(
            row,
            threshold,
            minimum_supported_harmonics=PARAMETERS.hum_min_supported_harmonics,
        )
        calibrated["qadd_mains_hum_null_reference_window_count"] = reference_count
        if reference_count is None:
            calibrated["qadd_mains_hum_null_calibration_status"] = "not_applicable_insufficient_support"
        else:
            calibrated["qadd_mains_hum_null_calibration_status"] = (
                "applied_exact_count"
                if reference_count == count
                else "applied_conservative_support_bin"
            )
        calibrated_rows.append(calibrated)
    recording_table = pd.DataFrame(calibrated_rows).sort_values(
        "logical_recording_id"
    ).reset_index(drop=True)

    save_table(pool, VALIDATION / "qadd_v420_hum_null_window_pool")
    save_table(hum_null_grid, VALIDATION / "qadd_v420_hum_null_calibration_grid")
    save_table(recording_table, TABLES / "qadd_v420_recording_features")

    extraction_checks = pd.DataFrame(
        [
            {
                "gate": "G1",
                "check": "one recording row per eligible ID",
                "passed": (
                    len(recording_table) == len(frozen)
                    and not recording_table["logical_recording_id"].duplicated().any()
                    and set(recording_table["logical_recording_id"])
                    == set(frozen["logical_recording_id"].astype(str))
                ),
                "observed": len(recording_table),
                "required": len(frozen),
            },
            {
                "gate": "G1",
                "check": "all local media hashes match the frozen media hashes",
                "passed": bool(
                    recording_table["media_sha256_verified"].fillna(False).astype(bool).all()
                ),
                "observed": int(
                    recording_table["media_sha256_verified"].fillna(False).astype(bool).sum()
                ),
                "required": len(recording_table),
            },
            {
                "gate": "G2",
                "check": "all recording-level raw estimands reconstruct from saved ledgers",
                "passed": len(extraction_errors) == 0,
                "observed": f"errors={len(extraction_errors)}",
                "required": "0",
            },
            {
                "gate": "G6",
                "check": "hum null calibration includes every observed eligible window count",
                "passed": set(support_grid)
                == set(
                    count
                    for count in observed_hum_window_counts
                    if int(count) >= 2
                ),
                "observed": support_grid,
                "required": "all unique observed counts >= 2",
            },
            {
                "gate": "G6",
                "check": "count-matched hum null grid is non-increasing with support",
                "passed": bool(
                    (np.diff(hum_null_grid["null_p95_db"].to_numpy(float)) <= 1e-12).all()
                ),
                "observed": hum_null_grid["null_p95_db"].to_list(),
                "required": "non-increasing",
            },
            {
                "gate": "G6",
                "check": "unavailable values remain NaN rather than zero-imputed",
                "passed": all(
                    recording_table.loc[
                        ~recording_table[f"{feature}_status"].astype(str).str.startswith("ok_"),
                        feature,
                    ].isna().all()
                    for feature in ANALYSIS_FEATURES
                ),
                "observed": "checked all five features",
                "required": "all unavailable values are NaN",
            },
            {
                "gate": "G7",
                "check": "spectral flatness remains within its mathematical range",
                "passed": recording_table[
                    "qadd_pause_spectral_flatness"
                ].dropna().between(0, 1).all(),
                "observed": "checked",
                "required": "[0,1]",
            },
            {
                "gate": "G7",
                "check": "no scalar QADD score was constructed",
                "passed": not any(
                    column in recording_table
                    for column in ["qadd_score", "qadd_composite", "qadd_burden"]
                ),
                "observed": "absent",
                "required": "absent",
            },
        ]
    )
    save_table(extraction_checks, VALIDATION / "qadd_v420_cohort_extraction_checks")
    display(extraction_checks)
else:
    pool = pd.DataFrame()
    hum_null_grid = pd.DataFrame()
    extraction_checks = pd.DataFrame(
        [
            {
                "gate": "G1",
                "check": "corrected cohort extraction completed",
                "passed": False,
                "observed": "NOT RUN",
                "required": "completed",
            }
        ]
    )


In [ ]:
reconstruction_audit_parts = []
deletion_parts = []
ledger_inventory_rows = []

if RUN_COHORT_EXTRACTION and len(recording_table):
    for recording_id in recording_table["logical_recording_id"].astype(str):
        paths = ledger_checkpoint_paths(recording_id)
        reconstruction = pd.read_csv(paths["reconstruction"])
        reconstruction_audit_parts.append(reconstruction)
        if paths["deletion"].exists():
            deletion_parts.append(pd.read_parquet(paths["deletion"]))
        for kind in ["frame", "interval", "spectral", "reconstruction"]:
            path = paths[kind]
            if path.exists():
                ledger_inventory_rows.append(
                    {
                        "logical_recording_id": recording_id,
                        "ledger_kind": kind,
                        "relative_path": str(path.relative_to(STAGE)).replace("\\", "/"),
                        "bytes": path.stat().st_size,
                        "sha256": sha256_file(path),
                    }
                )

reconstruction_audit = (
    pd.concat(reconstruction_audit_parts, ignore_index=True)
    if reconstruction_audit_parts
    else pd.DataFrame()
)
cluster_deletion = (
    pd.concat(deletion_parts, ignore_index=True)
    if deletion_parts
    else pd.DataFrame()
)
cluster_summary = (
    summarize_cluster_deletion(cluster_deletion)
    if len(cluster_deletion)
    else pd.DataFrame()
)
ledger_inventory = pd.DataFrame(ledger_inventory_rows)

if len(reconstruction_audit):
    save_table(reconstruction_audit, VALIDATION / "qadd_v420_reconstruction_audit")
if len(cluster_deletion):
    save_table(cluster_deletion, VALIDATION / "qadd_v420_pause_delete_one_long")
if len(cluster_summary):
    save_table(cluster_summary, VALIDATION / "qadd_v420_pause_delete_one_by_recording")
save_table(ledger_inventory, LEDGERS / "qadd_v420_ledger_inventory", parquet=False)

legacy_path = (
    LEGACY_MAIN
    / "02_FEATURE_FREEZE"
    / "additive_interference"
    / "qadd-v4.1.0"
    / "tables"
    / "qadd_v4_1_recording_features.csv"
)
migration = pd.DataFrame()
if RUN_COHORT_EXTRACTION and legacy_path.exists() and len(recording_table):
    legacy = pd.read_csv(legacy_path)
    comparison = recording_table.merge(
        legacy[["logical_recording_id", *ANALYSIS_FEATURES]],
        on="logical_recording_id",
        suffixes=("_v420", "_v410"),
        how="inner",
        validate="one_to_one",
    )
    rows = []
    for feature in ANALYSIS_FEATURES:
        delta = (
            pd.to_numeric(comparison[f"{feature}_v420"], errors="coerce")
            - pd.to_numeric(comparison[f"{feature}_v410"], errors="coerce")
        )
        finite = delta[np.isfinite(delta)]
        rows.append(
            {
                "feature": feature,
                "n_compared": len(finite),
                "median_signed_delta": float(finite.median()) if len(finite) else np.nan,
                "median_absolute_delta": float(finite.abs().median()) if len(finite) else np.nan,
                "p95_absolute_delta": float(finite.abs().quantile(0.95)) if len(finite) else np.nan,
                "maximum_absolute_delta": float(finite.abs().max()) if len(finite) else np.nan,
            }
        )
    migration = pd.DataFrame(rows)
    save_table(migration, VALIDATION / "qadd_v420_migration_from_v410")
display(migration)


In [ ]:
boundary_rows = []
robustness_error_rows = []

if (
    RUN_COHORT_EXTRACTION
    and RUN_COHORT_ROBUSTNESS
    and len(recording_table)
):
    eligible_boundary = recording_table.loc[
        recording_table["qadd_pause_ac_level_dbfs_median_raw_estimate"].notna()
    ].copy()
    boundary_sample = select_evenly(
        eligible_boundary,
        min(MAX_BOUNDARY_RECORDINGS, len(eligible_boundary)),
        sort_columns=[
            "qadd_pause_effective_nonfloor_support_sec",
            "qadd_pause_ac_level_dbfs_median_raw_estimate",
            "logical_recording_id",
        ],
    )
    frozen_by_id = frozen.set_index("logical_recording_id", drop=False)
    ffmpeg = shutil.which("ffmpeg")
    ffprobe = shutil.which("ffprobe")

    for position, recording_id in enumerate(
        boundary_sample["logical_recording_id"].astype(str), start=1
    ):
        try:
            source_row = frozen_by_id.loc[recording_id]
            media_path = resolve_media_path(
                source_row["media_path"],
                media_root_override=(
                    Path(MEDIA_ROOT_OVERRIDE) if MEDIA_ROOT_OVERRIDE is not None else None
                ),
                media_path_map=MEDIA_PATH_MAP,
            )
            views = decode_audio_views(
                media_path, ffmpeg=ffmpeg, ffprobe=ffprobe, analysis_rate=FS
            )
            primary, _ = intervals_for_recording(
                canonical_tables[CANONICAL_PRIMARY_VIEW], recording_id
            )
            strict_speech, _ = intervals_for_recording(
                canonical_tables[CANONICAL_SPEECH_VIEW], recording_id
            )
            strict_pause, _ = intervals_for_recording(
                canonical_tables[CANONICAL_PAUSE_VIEW], recording_id
            )
            reference = extract_qadd(
                views.analysis_16k,
                FS,
                primary_speech=primary,
                strict_speech=strict_speech,
                strict_internal_nonspeech=strict_pause,
                logical_recording_id=recording_id,
                speech_intervals_are_guarded=True,
                pause_intervals_are_guarded=True,
                parameters=PARAMETERS,
            ).recording

            alternatives = {}
            duration_sec = len(views.analysis_16k) / FS
            for extra_pause_ms in [100.0, 200.0]:
                eroded_pause = erode_intervals(
                    strict_pause,
                    duration_sec,
                    guard_ms=extra_pause_ms,
                    minimum_ms=PARAMETERS.minimum_residual_pause_ms,
                )
                alternatives[f"pause_extra_{int(extra_pause_ms)}ms"] = extract_qadd(
                    views.analysis_16k,
                    FS,
                    primary_speech=primary,
                    strict_speech=strict_speech,
                    strict_internal_nonspeech=eroded_pause,
                    logical_recording_id=recording_id,
                    speech_intervals_are_guarded=True,
                    pause_intervals_are_guarded=True,
                    parameters=PARAMETERS,
                ).recording

            extra_speech = erode_intervals(
                strict_speech,
                duration_sec,
                guard_ms=50.0,
                minimum_ms=PARAMETERS.frame_ms,
            )
            alternatives["speech_extra_50ms"] = extract_qadd(
                views.analysis_16k,
                FS,
                primary_speech=primary,
                strict_speech=extra_speech,
                strict_internal_nonspeech=strict_pause,
                logical_recording_id=recording_id,
                speech_intervals_are_guarded=True,
                pause_intervals_are_guarded=True,
                parameters=PARAMETERS,
            ).recording

            for condition, alternative in alternatives.items():
                for feature in ANALYSIS_FEATURES:
                    reference_value = reference.get(feature, np.nan)
                    alternative_value = alternative.get(feature, np.nan)
                    boundary_rows.append(
                        {
                            "logical_recording_id": recording_id,
                            "condition": condition,
                            "feature": feature,
                            "reference": reference_value,
                            "alternative": alternative_value,
                            "absolute_change": (
                                abs(alternative_value - reference_value)
                                if np.isfinite(reference_value)
                                and np.isfinite(alternative_value)
                                else np.nan
                            ),
                            "reference_available": np.isfinite(reference_value),
                            "alternative_available": np.isfinite(alternative_value),
                            "availability_changed": (
                                np.isfinite(reference_value)
                                != np.isfinite(alternative_value)
                            ),
                            "reference_pause_support_sec": reference[
                                "qadd_pause_effective_nonfloor_support_sec"
                            ],
                            "alternative_pause_support_sec": alternative[
                                "qadd_pause_effective_nonfloor_support_sec"
                            ],
                            "reference_speech_support_sec": reference[
                                "qadd_speech_effective_nonfloor_support_sec"
                            ],
                            "alternative_speech_support_sec": alternative[
                                "qadd_speech_effective_nonfloor_support_sec"
                            ],
                        }
                    )
        except Exception as exc:
            robustness_error_rows.append(
                {
                    "logical_recording_id": recording_id,
                    "error_type": type(exc).__name__,
                    "message": str(exc),
                }
            )
        if position % 20 == 0 or position == len(boundary_sample):
            print(f"Boundary audit [{position}/{len(boundary_sample)}]")

boundary_sensitivity = pd.DataFrame(boundary_rows)
robustness_errors = pd.DataFrame(
    robustness_error_rows,
    columns=["logical_recording_id", "error_type", "message"],
)
if len(boundary_sensitivity):
    save_table(boundary_sensitivity, VALIDATION / "qadd_v420_boundary_sensitivity")
save_table(robustness_errors, AUDIT / "qadd_v420_robustness_errors", parquet=False)

cohort_iqr = {
    feature: finite_iqr(recording_table[feature])
    for feature in ANALYSIS_FEATURES
} if len(recording_table) else {}

cluster_population_rows = []
if len(cluster_summary):
    for feature, local in cluster_summary.groupby("feature", sort=False):
        values = pd.to_numeric(
            local["delete_one_p90_absolute_change"], errors="coerce"
        ).dropna()
        scale = cohort_iqr.get(feature, np.nan)
        relative = values / scale if np.isfinite(scale) and scale > 0 else np.nan
        cluster_population_rows.append(
            {
                "feature": feature,
                "recording_count": len(values),
                "cohort_iqr": scale,
                "recording_p90_change_median": float(values.median()) if len(values) else np.nan,
                "recording_p90_change_population_p90": float(values.quantile(0.90)) if len(values) else np.nan,
                "relative_to_cohort_iqr_median": float(pd.Series(relative).median()) if len(values) else np.nan,
                "relative_to_cohort_iqr_population_p90": float(pd.Series(relative).quantile(0.90)) if len(values) else np.nan,
            }
        )
cluster_population = pd.DataFrame(cluster_population_rows)
if len(cluster_population):
    save_table(cluster_population, VALIDATION / "qadd_v420_pause_delete_one_population")

boundary_population_rows = []
if len(boundary_sensitivity):
    for (condition, feature), local in boundary_sensitivity.groupby(
        ["condition", "feature"], sort=False
    ):
        values = pd.to_numeric(local["absolute_change"], errors="coerce").dropna()
        scale = cohort_iqr.get(feature, np.nan)
        relative = values / scale if np.isfinite(scale) and scale > 0 else np.nan
        boundary_population_rows.append(
            {
                "condition": condition,
                "feature": feature,
                "recording_count": len(local),
                "finite_change_count": len(values),
                "cohort_iqr": scale,
                "absolute_change_median": float(values.median()) if len(values) else np.nan,
                "absolute_change_p95": float(values.quantile(0.95)) if len(values) else np.nan,
                "relative_to_cohort_iqr_median": float(pd.Series(relative).median()) if len(values) else np.nan,
                "relative_to_cohort_iqr_p95": float(pd.Series(relative).quantile(0.95)) if len(values) else np.nan,
                "availability_change_fraction": float(local["availability_changed"].mean()),
            }
        )
boundary_population = pd.DataFrame(boundary_population_rows)
if len(boundary_population):
    save_table(boundary_population, VALIDATION / "qadd_v420_boundary_sensitivity_population")

g6_cohort_checks = pd.DataFrame(
    [
        {
            "gate": "G6",
            "check": "whole-pause deletion audit covers all five features",
            "passed": set(cluster_population.get("feature", [])) == set(ANALYSIS_FEATURES),
            "observed": sorted(set(cluster_population.get("feature", []))),
            "required": sorted(ANALYSIS_FEATURES),
        },
        {
            "gate": "G6",
            "check": "pause and speech boundary sensitivity covers all five features",
            "passed": (
                set(boundary_sensitivity.get("feature", [])) == set(ANALYSIS_FEATURES)
                if len(boundary_sensitivity)
                else False
            ),
            "observed": sorted(set(boundary_sensitivity.get("feature", []))),
            "required": sorted(ANALYSIS_FEATURES),
        },
        {
            "gate": "G6",
            "check": "cohort robustness completed without processing errors",
            "passed": robustness_errors.empty,
            "observed": len(robustness_errors),
            "required": 0,
        },
        {
            "gate": "G6",
            "check": "quantitative robustness evidence is complete for scientific review",
            "passed": bool(len(cluster_population) and len(boundary_population)),
            "observed": f"delete-one rows={len(cluster_population)}; boundary rows={len(boundary_population)}",
            "required": "both summaries present",
        },
    ]
)
save_table(g6_cohort_checks, VALIDATION / "qadd_v420_g6_cohort_checks")
display(cluster_population)
display(boundary_population)
display(g6_cohort_checks)


In [ ]:
if RUN_COHORT_EXTRACTION and len(recording_table):
    empirical_summary = empirical_feature_summary(recording_table)
    correlations = recording_table[list(ANALYSIS_FEATURES)].corr(
        method="spearman", min_periods=20
    )
    support_tiers = []
    tier_fields = {
        "qadd_pause_ac_level_dbfs_median": "qadd_pause_level_support_tier",
        "qadd_pause_level_iqr_db": "qadd_pause_dispersion_support_tier",
        "qadd_speech_pause_level_contrast_db": "qadd_speech_pause_contrast_support_tier",
        "qadd_pause_spectral_flatness": "qadd_flatness_support_tier",
        "qadd_mains_hum_comb_score_db": "qadd_hum_support_tier",
    }
    for feature, field in tier_fields.items():
        counts = recording_table[field].astype(str).value_counts(dropna=False)
        for tier, count in counts.items():
            support_tiers.append(
                {
                    "feature": feature,
                    "support_tier": tier,
                    "recording_count": int(count),
                    "recording_fraction": float(count / len(recording_table)),
                }
            )
    support_tier_table = pd.DataFrame(support_tiers)

    hum_joint = recording_table[
        "qadd_mains_hum_joint_evidence_above_null"
    ].map(
        lambda value: value
        if isinstance(value, (bool, np.bool_))
        else str(value).strip().lower() == "true"
    )
    hum_eligible = recording_table[
        "qadd_mains_hum_null_calibration_status"
    ].astype(str).str.startswith("applied")
    hum_summary = pd.DataFrame(
        [
            {
                "eligible_recordings": int(hum_eligible.sum()),
                "joint_evidence_recordings": int(
                    (hum_joint & hum_eligible).sum()
                ),
                "joint_evidence_fraction_among_eligible": (
                    float((hum_joint & hum_eligible).sum() / hum_eligible.sum())
                    if hum_eligible.sum()
                    else np.nan
                ),
                "winner_50_count": int(
                    recording_table["qadd_mains_hum_winner_hz"].eq(50).sum()
                ),
                "winner_60_count": int(
                    recording_table["qadd_mains_hum_winner_hz"].eq(60).sum()
                ),
            }
        ]
    )

    save_table(empirical_summary, TABLES / "qadd_v420_empirical_summary")
    save_table(
        correlations.reset_index(names="feature"),
        TABLES / "qadd_v420_spearman_correlations",
    )
    save_table(support_tier_table, TABLES / "qadd_v420_support_tier_counts")
    save_table(hum_summary, TABLES / "qadd_v420_hum_joint_evidence_summary")

    g7_checks = pd.DataFrame(
        [
            {
                "gate": "G7",
                "check": "one empirical summary row exists per retained feature",
                "passed": len(empirical_summary) == len(ANALYSIS_FEATURES),
                "observed": len(empirical_summary),
                "required": len(ANALYSIS_FEATURES),
            },
            {
                "gate": "G7",
                "check": "feature availability and status distributions are explicit",
                "passed": empirical_summary["available_fraction"].between(0, 1).all(),
                "observed": empirical_summary[
                    ["feature", "available_fraction"]
                ].to_dict(orient="records"),
                "required": "[0,1] for every feature",
            },
            {
                "gate": "G7",
                "check": "hum-like joint evidence prevalence is reported separately from the raw descriptor",
                "passed": len(hum_summary) == 1,
                "observed": hum_summary.to_dict(orient="records"),
                "required": "reported",
            },
            {
                "gate": "G7",
                "check": "empirical distributions contain no infinite values",
                "passed": all(
                    np.isfinite(
                        pd.to_numeric(recording_table[feature], errors="coerce")
                        .dropna()
                        .to_numpy(float)
                    ).all()
                    for feature in ANALYSIS_FEATURES
                ),
                "observed": "checked all five features",
                "required": "finite or missing",
            },
        ]
    )
    save_table(g7_checks, VALIDATION / "qadd_v420_g7_checks")
else:
    empirical_summary = pd.DataFrame()
    correlations = pd.DataFrame()
    support_tier_table = pd.DataFrame()
    hum_summary = pd.DataFrame()
    g7_checks = pd.DataFrame()
display(empirical_summary)
display(hum_summary)


In [ ]:
if RUN_COHORT_EXTRACTION and len(recording_table):
    persistence = repeated_recording_persistence(
        recording_table,
        subject_column=subject_column,
        date_column=date_column,
    )
    balanced_draws = participant_balanced_resampling(
        recording_table,
        subject_column=subject_column,
        iterations=PARTICIPANT_BALANCED_ITERATIONS,
        seed=PARAMETERS.random_seed + 201,
    )
    balanced_summary = participant_balanced_summary(balanced_draws)

    recording_weighted = empirical_summary[
        ["feature", "median"]
    ].rename(columns={"median": "recording_weighted_median"})
    weighting_comparison = recording_weighted.merge(
        balanced_summary, on="feature", how="left", validate="one_to_one"
    )
    weighting_comparison["balanced_minus_recording_weighted"] = (
        weighting_comparison["balanced_median_of_medians"]
        - weighting_comparison["recording_weighted_median"]
    )

    correlation_pairs = []
    for left_index, left in enumerate(ANALYSIS_FEATURES):
        for right in ANALYSIS_FEATURES[left_index + 1 :]:
            value = correlations.loc[left, right]
            correlation_pairs.append(
                {
                    "feature_left": left,
                    "feature_right": right,
                    "spearman": value,
                    "absolute_spearman": abs(value) if np.isfinite(value) else np.nan,
                }
            )
    correlation_pair_table = pd.DataFrame(correlation_pairs)
    maximum_redundancy = (
        float(correlation_pair_table["absolute_spearman"].max())
        if len(correlation_pair_table)
        else np.nan
    )

    save_table(persistence, TABLES / "qadd_v420_repeated_recording_persistence")
    save_table(balanced_draws, TABLES / "qadd_v420_participant_balanced_draws")
    save_table(balanced_summary, TABLES / "qadd_v420_participant_balanced_summary")
    save_table(weighting_comparison, TABLES / "qadd_v420_weighting_comparison")
    save_table(correlation_pair_table, TABLES / "qadd_v420_correlation_pairs")

    g8_checks = pd.DataFrame(
        [
            {
                "gate": "G8",
                "check": "repeated-recording persistence is reported for every retained feature",
                "passed": set(persistence["feature"]) == set(ANALYSIS_FEATURES),
                "observed": len(persistence),
                "required": len(ANALYSIS_FEATURES),
            },
            {
                "gate": "G8",
                "check": "participant-balanced resampling completed",
                "passed": (
                    len(balanced_draws)
                    == PARTICIPANT_BALANCED_ITERATIONS * len(ANALYSIS_FEATURES)
                ),
                "observed": len(balanced_draws),
                "required": PARTICIPANT_BALANCED_ITERATIONS * len(ANALYSIS_FEATURES),
            },
            {
                "gate": "G8",
                "check": "within-family redundancy is quantified rather than assumed",
                "passed": len(correlation_pair_table)
                == len(ANALYSIS_FEATURES) * (len(ANALYSIS_FEATURES) - 1) / 2,
                "observed": maximum_redundancy,
                "required": "all unique feature pairs",
            },
        ]
    )
    save_table(g8_checks, VALIDATION / "qadd_v420_g8_checks")
else:
    persistence = pd.DataFrame()
    balanced_draws = pd.DataFrame()
    balanced_summary = pd.DataFrame()
    weighting_comparison = pd.DataFrame()
    correlation_pair_table = pd.DataFrame()
    g8_checks = pd.DataFrame()
display(persistence)
display(weighting_comparison)


In [ ]:
if RUN_COHORT_EXTRACTION and len(recording_table):
    measurements_long = measurement_long_frame(recording_table)
    model_ready = model_interface_frame(recording_table)
    save_table(measurements_long, TABLES / "qadd_v420_measurements_long")
    save_table(model_ready, TABLES / "qadd_v420_model_interface")

    ml_checks = pd.DataFrame(
        [
            {
                "gate": "ML",
                "check": "each value is accompanied by availability, status, and support tier",
                "passed": all(
                    all(
                        column in model_ready
                        for column in [
                            feature,
                            f"{feature}__available",
                            f"{feature}__status",
                            f"{feature}__support_tier",
                        ]
                    )
                    for feature in ANALYSIS_FEATURES
                ),
                "observed": len(model_ready.columns),
                "required": "complete companions for five features",
            },
            {
                "gate": "ML",
                "check": "missing measurements are not imputed",
                "passed": all(
                    model_ready.loc[
                        ~model_ready[f"{feature}__available"].astype(bool),
                        feature,
                    ].isna().all()
                    for feature in ANALYSIS_FEATURES
                ),
                "observed": "checked",
                "required": "missing stays NaN",
            },
            {
                "gate": "ML",
                "check": "no standalone reject threshold or family scalar is exposed",
                "passed": (
                    model_ready["qadd_standalone_reject_allowed"].eq(False).all()
                    and model_ready["qadd_family_scalar_available"].eq(False).all()
                ),
                "observed": "false/false",
                "required": "false/false",
            },
        ]
    )
    save_table(ml_checks, VALIDATION / "qadd_v420_ml_handoff_checks")
else:
    measurements_long = pd.DataFrame()
    model_ready = pd.DataFrame()
    ml_checks = pd.DataFrame()
display(ml_checks)


In [ ]:
figure_index_rows = []

if RUN_COHORT_EXTRACTION and len(recording_table):
    feature_labels = {
        "qadd_pause_ac_level_dbfs_median": "Pause level",
        "qadd_pause_level_iqr_db": "Pause-level IQR",
        "qadd_speech_pause_level_contrast_db": "Speech–pause contrast",
        "qadd_pause_spectral_flatness": "Spectral flatness",
        "qadd_mains_hum_comb_score_db": "Hum-comb score",
    }

    # Panel D — support and availability.
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    availability = empirical_summary[["feature", "available_fraction"]].copy()
    axes[0, 0].bar(
        [feature_labels[item] for item in availability["feature"]],
        availability["available_fraction"],
    )
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].set_ylabel("Available fraction")
    axes[0, 0].tick_params(axis="x", rotation=35)
    axes[0, 0].set_title("Feature availability")

    tier_order = ["unavailable", "minimum", "moderate", "high"]
    pivot = (
        support_tier_table.pivot_table(
            index="feature",
            columns="support_tier",
            values="recording_fraction",
            aggfunc="sum",
            fill_value=0,
        )
        .reindex(index=ANALYSIS_FEATURES, fill_value=0)
        .reindex(columns=tier_order, fill_value=0)
    )
    bottom = np.zeros(len(pivot))
    for tier in tier_order:
        axes[0, 1].bar(
            [feature_labels[item] for item in pivot.index],
            pivot[tier].to_numpy(float),
            bottom=bottom,
            label=tier,
        )
        bottom += pivot[tier].to_numpy(float)
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].set_ylabel("Recording fraction")
    axes[0, 1].tick_params(axis="x", rotation=35)
    axes[0, 1].set_title("Support tiers")
    axes[0, 1].legend(fontsize=8)

    axes[1, 0].hist(
        pd.to_numeric(
            recording_table["qadd_pause_effective_nonfloor_support_sec"],
            errors="coerce",
        ).dropna(),
        bins="fd",
    )
    axes[1, 0].set_xlabel("Effective non-floor pause support (s)")
    axes[1, 0].set_ylabel("Recordings")
    axes[1, 0].set_title("Pause support")

    floor_fraction = pd.to_numeric(
        recording_table["qadd_pause_at_floor_frame_fraction"], errors="coerce"
    )
    axes[1, 1].hist(floor_fraction.dropna(), bins="fd")
    axes[1, 1].axvline(
        PARAMETERS.maximum_floor_censored_fraction,
        linestyle="--",
        label="censoring ceiling",
    )
    axes[1, 1].set_xlabel("Pause frames at computational floor")
    axes[1, 1].set_ylabel("Recordings")
    axes[1, 1].set_title("Digital-floor mixture")
    axes[1, 1].legend(fontsize=8)
    fig.tight_layout()
    d_source = pd.concat(
        [
            availability.assign(section="availability"),
            support_tier_table.assign(section="support_tier"),
            recording_table[
                [
                    "logical_recording_id",
                    "qadd_pause_effective_nonfloor_support_sec",
                    "qadd_pause_at_floor_frame_fraction",
                    "qadd_flatness_valid_window_count",
                    "qadd_hum_valid_window_count",
                ]
            ].assign(section="recording_support"),
        ],
        ignore_index=True,
        sort=False,
    )
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="D_support_and_availability",
            source_data=d_source,
            caption=(
                "Panel D. QADD feature availability, feature-specific support tiers, "
                "effective non-floor guarded-pause support, and digital-floor mixtures. "
                "Unavailable values remain missing and are not replaced with zero."
            ),
            provenance={
                "panel": "D",
                "evidence_type": "cohort support and availability",
                "recording_count": len(recording_table),
            },
        )
    )
    plt.close(fig)

    # Panel E — pause deletion and boundary sensitivity, scaled by cohort IQR.
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    if len(cluster_population):
        x = np.arange(len(cluster_population))
        axes[0].bar(
            x,
            cluster_population["relative_to_cohort_iqr_population_p90"],
        )
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(
            [feature_labels[item] for item in cluster_population["feature"]],
            rotation=35,
            ha="right",
        )
    axes[0].set_ylabel("Population P90 change / cohort IQR")
    axes[0].set_title("Delete one whole pause")

    if len(boundary_population):
        conditions = list(boundary_population["condition"].drop_duplicates())
        width = 0.8 / max(1, len(conditions))
        feature_positions = np.arange(len(ANALYSIS_FEATURES))
        for condition_index, condition in enumerate(conditions):
            local = (
                boundary_population.loc[
                    boundary_population["condition"].eq(condition)
                ]
                .set_index("feature")
                .reindex(ANALYSIS_FEATURES)
            )
            axes[1].bar(
                feature_positions
                + (condition_index - (len(conditions) - 1) / 2) * width,
                local["relative_to_cohort_iqr_p95"],
                width=width,
                label=condition,
            )
        axes[1].set_xticks(feature_positions)
        axes[1].set_xticklabels(
            [feature_labels[item] for item in ANALYSIS_FEATURES],
            rotation=35,
            ha="right",
        )
        axes[1].legend(fontsize=8)
    axes[1].set_ylabel("P95 absolute change / cohort IQR")
    axes[1].set_title("Boundary perturbations")
    fig.tight_layout()
    e_source = pd.concat(
        [
            cluster_population.assign(section="delete_one"),
            boundary_population.assign(section="boundary"),
        ],
        ignore_index=True,
        sort=False,
    )
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="E_support_boundary_sensitivity",
            source_data=e_source,
            caption=(
                "Panel E. Whole-pause deletion and additional pause/speech boundary "
                "erosion sensitivity. Changes are normalized by each feature's cohort "
                "IQR to permit comparison across unlike units. These are diagnostics, "
                "not independent observations."
            ),
            provenance={
                "panel": "E",
                "evidence_type": "clustered support and boundary sensitivity",
                "boundary_recording_count": int(
                    boundary_sensitivity["logical_recording_id"].nunique()
                    if len(boundary_sensitivity)
                    else 0
                ),
            },
        )
    )
    plt.close(fig)

    # Panel F — empirical feature distributions.
    fig, axes = plt.subplots(2, 3, figsize=(13, 7.6))
    distribution_rows = []
    for axis, feature in zip(axes.flat, ANALYSIS_FEATURES):
        values = pd.to_numeric(recording_table[feature], errors="coerce").dropna()
        axis.hist(values, bins="fd")
        if len(values):
            axis.axvline(values.median(), linestyle="--")
        axis.set_title(feature_labels[feature])
        axis.set_ylabel("Recordings")
        axis.set_xlabel(
            "dBFS" if feature.endswith("dbfs_median")
            else "ratio" if feature.endswith("flatness")
            else "dB"
        )
        distribution_rows.extend(
            {
                "logical_recording_id": recording_id,
                "feature": feature,
                "value": value,
            }
            for recording_id, value in zip(
                recording_table.loc[values.index, "logical_recording_id"],
                values,
            )
        )
    family_counts = recording_table["qadd_family_status"].astype(str).value_counts()
    axes.flat[-1].bar(family_counts.index, family_counts.values)
    axes.flat[-1].tick_params(axis="x", rotation=35)
    axes.flat[-1].set_title("Primary-feature family status")
    axes.flat[-1].set_ylabel("Recordings")
    fig.tight_layout()
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="F_empirical_distributions",
            source_data=pd.DataFrame(distribution_rows),
            caption=(
                "Panel F. Empirical distributions of the five QADD recording-level "
                "features and primary-feature family status. Spectral flatness is "
                "nonordinal; higher or lower values are not universally worse."
            ),
            provenance={
                "panel": "F",
                "evidence_type": "empirical cohort distributions",
                "recording_count": len(recording_table),
            },
        )
    )
    plt.close(fig)

    # Panel H — persistence, redundancy, and participant weighting.
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    x = np.arange(len(persistence))
    axes[0].bar(x, persistence["first_second_spearman"])
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(
        [feature_labels[item] for item in persistence["feature"]],
        rotation=35,
        ha="right",
    )
    axes[0].set_ylim(-1, 1)
    axes[0].set_ylabel("First–second Spearman")
    axes[0].set_title("Repeated-recording persistence")

    image = axes[1].imshow(
        correlations.reindex(index=ANALYSIS_FEATURES, columns=ANALYSIS_FEATURES),
        vmin=-1,
        vmax=1,
        cmap="coolwarm",
    )
    axes[1].set_xticks(np.arange(len(ANALYSIS_FEATURES)))
    axes[1].set_yticks(np.arange(len(ANALYSIS_FEATURES)))
    axes[1].set_xticklabels(
        [feature_labels[item] for item in ANALYSIS_FEATURES],
        rotation=45,
        ha="right",
    )
    axes[1].set_yticklabels([feature_labels[item] for item in ANALYSIS_FEATURES])
    axes[1].set_title("Within-family Spearman")
    fig.colorbar(image, ax=axes[1], fraction=0.046)

    axes[2].bar(
        np.arange(len(weighting_comparison)),
        weighting_comparison["balanced_minus_recording_weighted"].abs(),
    )
    axes[2].set_xticks(np.arange(len(weighting_comparison)))
    axes[2].set_xticklabels(
        [feature_labels[item] for item in weighting_comparison["feature"]],
        rotation=35,
        ha="right",
    )
    axes[2].set_ylabel("|Participant-balanced − recording-weighted median|")
    axes[2].set_title("Participant weighting sensitivity")
    fig.tight_layout()
    h_source = pd.concat(
        [
            persistence.assign(section="persistence"),
            correlations.reset_index(names="feature").assign(section="correlation"),
            weighting_comparison.assign(section="weighting"),
        ],
        ignore_index=True,
        sort=False,
    )
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="H_reliability_redundancy_weighting",
            source_data=h_source,
            caption=(
                "Panel H. Empirical repeated-recording persistence, within-family "
                "Spearman associations, and sensitivity of medians to participant "
                "versus recording weighting. Persistence is descriptive and does not "
                "assume a stable biological or acquisition state between sessions."
            ),
            provenance={
                "panel": "H",
                "evidence_type": "persistence, redundancy, participant weighting",
                "participant_count": int(
                    recording_table[subject_column].nunique(dropna=True)
                ),
            },
        )
    )
    plt.close(fig)

    # Panel J — ML handoff contract.
    ml_rows = []
    for feature in ANALYSIS_FEATURES:
        ml_rows.append(
            {
                "feature": feature,
                "value_present": feature in model_ready,
                "availability_present": f"{feature}__available" in model_ready,
                "status_present": f"{feature}__status" in model_ready,
                "support_present": f"{feature}__support_tier" in model_ready,
                "missing_values_imputed": bool(
                    model_ready.loc[
                        ~model_ready[f"{feature}__available"].astype(bool),
                        feature,
                    ].notna().any()
                ),
            }
        )
    ml_figure_data = pd.DataFrame(ml_rows)
    fig, axis = plt.subplots(figsize=(10, 4.5))
    components = [
        "value_present",
        "availability_present",
        "status_present",
        "support_present",
    ]
    matrix = ml_figure_data[components].astype(int).to_numpy()
    image = axis.imshow(matrix, vmin=0, vmax=1, cmap="Greys")
    axis.set_xticks(np.arange(len(components)))
    axis.set_xticklabels(
        ["value", "availability", "status", "support tier"]
    )
    axis.set_yticks(np.arange(len(ANALYSIS_FEATURES)))
    axis.set_yticklabels(
        [feature_labels[item] for item in ANALYSIS_FEATURES]
    )
    axis.set_title("Quality-aware ML handoff completeness")
    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            axis.text(
                column_index,
                row_index,
                "✓" if matrix[row_index, column_index] else "×",
                ha="center",
                va="center",
            )
    fig.tight_layout()
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="J_ml_handoff_contract",
            source_data=ml_figure_data,
            caption=(
                "Panel J. QADD model-facing export contract. Every feature is "
                "accompanied by availability, status, and support tier; missing "
                "measurements remain missing. No family scalar or standalone rejection "
                "threshold is exposed."
            ),
            provenance={
                "panel": "J",
                "evidence_type": "quality-aware ML interface",
                "recording_count": len(model_ready),
            },
        )
    )
    plt.close(fig)

figure_index = pd.DataFrame(figure_index_rows)
if len(figure_index):
    save_table(figure_index, FIGURES / "qadd_v420_cohort_figure_index", parquet=False)
display(figure_index)


In [ ]:
gallery_index_rows = []
gallery_error_rows = []

if RUN_COHORT_EXTRACTION and BUILD_GALLERY and len(recording_table):
    candidates = []
    eligible = recording_table.loc[
        recording_table["qadd_pause_ac_level_dbfs_median_raw_estimate"].notna()
    ].copy()
    for quantile in [0.05, 0.25, 0.50, 0.75, 0.95]:
        target = eligible[
            "qadd_pause_ac_level_dbfs_median_raw_estimate"
        ].quantile(quantile)
        index = (
            eligible["qadd_pause_ac_level_dbfs_median_raw_estimate"] - target
        ).abs().idxmin()
        candidates.append(
            (
                eligible.loc[index, "logical_recording_id"],
                f"pause_level_q{int(quantile*100):02d}",
            )
        )
    extrema = [
        ("qadd_pause_level_iqr_db_raw_estimate", "max", "highest_pause_iqr"),
        ("qadd_pause_spectral_flatness_raw_estimate", "min", "lowest_flatness"),
        ("qadd_pause_spectral_flatness_raw_estimate", "max", "highest_flatness"),
        ("qadd_mains_hum_excess_over_null_p95_db", "max", "highest_hum_excess"),
        ("qadd_pause_effective_nonfloor_support_sec", "min", "lowest_pause_support"),
        ("qadd_pause_at_floor_frame_fraction", "max", "highest_floor_fraction"),
    ]
    for column, operation, reason in extrema:
        local = recording_table.loc[
            pd.to_numeric(recording_table[column], errors="coerce").notna()
        ]
        if len(local):
            index = (
                pd.to_numeric(local[column], errors="coerce").idxmax()
                if operation == "max"
                else pd.to_numeric(local[column], errors="coerce").idxmin()
            )
            candidates.append((local.loc[index, "logical_recording_id"], reason))

    gallery_selection = (
        pd.DataFrame(candidates, columns=["logical_recording_id", "selection_reason"])
        .groupby("logical_recording_id", as_index=False)["selection_reason"]
        .agg(";".join)
        .head(GALLERY_RECORDING_LIMIT)
    )

    frozen_by_id = frozen.set_index("logical_recording_id", drop=False)
    ffmpeg = shutil.which("ffmpeg")
    ffprobe = shutil.which("ffprobe")

    for row in gallery_selection.itertuples(index=False):
        recording_id = str(row.logical_recording_id)
        try:
            source_row = frozen_by_id.loc[recording_id]
            media_path = resolve_media_path(
                source_row["media_path"],
                media_root_override=(
                    Path(MEDIA_ROOT_OVERRIDE) if MEDIA_ROOT_OVERRIDE is not None else None
                ),
                media_path_map=MEDIA_PATH_MAP,
            )
            views = decode_audio_views(
                media_path, ffmpeg=ffmpeg, ffprobe=ffprobe, analysis_rate=FS
            )
            primary, _ = intervals_for_recording(
                canonical_tables[CANONICAL_PRIMARY_VIEW], recording_id
            )
            strict_speech, _ = intervals_for_recording(
                canonical_tables[CANONICAL_SPEECH_VIEW], recording_id
            )
            strict_pause, _ = intervals_for_recording(
                canonical_tables[CANONICAL_PAUSE_VIEW], recording_id
            )
            extraction = extract_qadd(
                views.analysis_16k,
                FS,
                primary_speech=primary,
                strict_speech=strict_speech,
                strict_internal_nonspeech=strict_pause,
                logical_recording_id=recording_id,
                speech_intervals_are_guarded=True,
                pause_intervals_are_guarded=True,
                parameters=PARAMETERS,
            )
            time_axis = np.arange(len(views.analysis_16k)) / FS
            stride = max(1, len(time_axis) // 12000)
            fig, axes = plt.subplots(3, 1, figsize=(11, 7.2), sharex=True)
            axes[0].plot(
                time_axis[::stride],
                views.analysis_16k[::stride],
                linewidth=0.5,
            )
            for interval in strict_speech:
                axes[0].axvspan(
                    interval.start_sec, interval.end_sec, alpha=0.16
                )
            for interval in strict_pause:
                axes[0].axvspan(
                    interval.start_sec, interval.end_sec, alpha=0.28
                )
            axes[0].set_ylabel("Amplitude")
            axes[0].set_title(
                f"{recording_id} — {row.selection_reason}"
            )

            frame = extraction.frame_ledger
            for region, marker in [("speech", "."), ("pause", "x")]:
                local = frame.loc[frame["region"].eq(region)]
                axes[1].scatter(
                    (local["frame_start_sec"] + local["frame_end_sec"]) / 2,
                    local["rms_dbfs"],
                    s=7,
                    marker=marker,
                    label=region,
                )
            axes[1].axhline(
                PARAMETERS.dbfs_floor_db, linestyle=":", linewidth=1
            )
            axes[1].set_ylabel("30-ms AC level (dBFS)")
            axes[1].legend(fontsize=8)

            spectral = extraction.spectral_ledger
            flat = spectral.loc[
                spectral["window_kind"].eq("flatness")
                & pd.to_numeric(
                    spectral["spectral_flatness"], errors="coerce"
                ).notna()
            ]
            hum = spectral.loc[
                spectral["window_kind"].eq("hum")
                & pd.to_numeric(
                    spectral["hum_score_max_db"], errors="coerce"
                ).notna()
            ]
            if len(flat):
                axes[2].scatter(
                    (flat["window_start_sec"] + flat["window_end_sec"]) / 2,
                    flat["spectral_flatness"],
                    s=13,
                    label="spectral flatness",
                )
            axes[2].set_ylabel("Flatness")
            axes[2].set_ylim(0, 1)
            axes[2].set_xlabel("Recording time (s)")
            if len(hum):
                hum_axis = axes[2].twinx()
                hum_axis.scatter(
                    (hum["window_start_sec"] + hum["window_end_sec"]) / 2,
                    hum["hum_score_max_db"],
                    s=13,
                    marker="x",
                    label="hum-comb score",
                )
                hum_axis.set_ylabel("Hum-comb score (dB)")
            fig.tight_layout()

            stem = f"G_signal_example_{recording_id}"
            png = GALLERIES / f"{stem}.png"
            svg = GALLERIES / f"{stem}.svg"
            pdf = GALLERIES / f"{stem}.pdf"
            source = GALLERIES / f"{stem}.source.csv"
            caption = GALLERIES / f"{stem}.caption.md"
            provenance = GALLERIES / f"{stem}.provenance.json"
            fig.savefig(png, dpi=300, bbox_inches="tight")
            fig.savefig(svg, bbox_inches="tight")
            fig.savefig(pdf, bbox_inches="tight")
            pd.concat(
                [
                    frame.assign(source_table="frame"),
                    spectral.assign(source_table="spectral"),
                ],
                ignore_index=True,
                sort=False,
            ).to_csv(source, index=False)
            caption.write_text(
                (
                    f"Panel G signal-linked QADD audit example for {recording_id}; "
                    f"selection={row.selection_reason}. Speech and guarded pause "
                    "regions are frozen inputs. Frame levels, spectral flatness, "
                    "and hum-comb values are shown without human-QC or diagnosis "
                    "labels."
                ),
                encoding="utf-8",
            )
            write_json(
                {
                    "panel": "G",
                    "logical_recording_id": recording_id,
                    "selection_reason": row.selection_reason,
                    "measurement_version": MEASUREMENT_VERSION,
                    "media_sha256": sha256_file(media_path),
                    "source_csv_sha256": sha256_file(source),
                    "label_blind_selection": True,
                },
                provenance,
            )
            plt.close(fig)
            gallery_index_rows.append(
                {
                    "logical_recording_id": recording_id,
                    "selection_reason": row.selection_reason,
                    "png": str(png),
                    "svg": str(svg),
                    "pdf": str(pdf),
                    "source_csv": str(source),
                    "caption": str(caption),
                    "provenance": str(provenance),
                }
            )
        except Exception as exc:
            gallery_error_rows.append(
                {
                    "logical_recording_id": recording_id,
                    "error_type": type(exc).__name__,
                    "message": str(exc),
                }
            )

gallery_index = pd.DataFrame(gallery_index_rows)
gallery_errors = pd.DataFrame(
    gallery_error_rows,
    columns=["logical_recording_id", "error_type", "message"],
)
if len(gallery_index):
    save_table(gallery_index, GALLERIES / "qadd_v420_gallery_index", parquet=False)
save_table(gallery_errors, AUDIT / "qadd_v420_gallery_errors", parquet=False)
display(gallery_index)
display(gallery_errors)


In [ ]:
feature_decisions = pd.DataFrame(
    [
        {
            "feature": "qadd_pause_ac_level_dbfs_median",
            "provisional_role": "primary contextual acquisition measurement",
            "g10_decision": "PENDING_COHORT_REVIEW",
            "claim": "typical recorded non-floor guarded-pause energy in analysis-view dBFS",
            "standalone_gate_allowed": False,
        },
        {
            "feature": "qadd_pause_level_iqr_db",
            "provisional_role": "secondary nonstationarity descriptor",
            "g10_decision": "PENDING_COHORT_REVIEW",
            "claim": "within-recording dispersion of guarded-pause frame levels",
            "standalone_gate_allowed": False,
        },
        {
            "feature": "qadd_speech_pause_level_contrast_db",
            "provisional_role": "secondary mixed speech/acquisition descriptor",
            "g10_decision": "PENDING_COHORT_REVIEW",
            "claim": "within-recording speech-to-pause level separation; not physical SNR",
            "standalone_gate_allowed": False,
        },
        {
            "feature": "qadd_pause_spectral_flatness",
            "provisional_role": "secondary nonordinal spectral-type descriptor",
            "g10_decision": "PENDING_COHORT_REVIEW",
            "claim": "broadband-like versus tonal/structured pause spectrum",
            "standalone_gate_allowed": False,
        },
        {
            "feature": "qadd_mains_hum_comb_score_db",
            "provisional_role": "targeted hum-like structure descriptor",
            "g10_decision": "PENDING_COHORT_REVIEW",
            "claim": "50/60-Hz harmonic prominence; not unique source identity",
            "standalone_gate_allowed": False,
        },
    ]
)
save_table(feature_decisions, VALIDATION / "qadd_v420_g10_feature_decisions")

cohort_complete = bool(
    RUN_COHORT_EXTRACTION
    and len(recording_table) == len(frozen)
    and extraction_errors.empty
)
required_cohort_panels = {
    "D_support_and_availability",
    "E_support_boundary_sensitivity",
    "F_empirical_distributions",
    "H_reliability_redundancy_weighting",
    "J_ml_handoff_contract",
}
completed_cohort_panels = set(figure_index.get("panel_stem", pd.Series(dtype=str)))
gallery_complete = bool(
    not BUILD_GALLERY or (len(gallery_index) >= 8 and gallery_errors.empty)
)

cohort_gate_rows = []
for frame in [
    g1_input_checks,
    extraction_checks,
    g6_cohort_checks,
    g7_checks,
    g8_checks,
    ml_checks,
]:
    if len(frame):
        cohort_gate_rows.append(frame)
cohort_checks = (
    pd.concat(cohort_gate_rows, ignore_index=True, sort=False)
    if cohort_gate_rows
    else pd.DataFrame()
)
save_table(cohort_checks, VALIDATION / "qadd_v420_cohort_checks")

gate_summary = preflight_gates.copy()
gate_summary["status"] = gate_summary["status"].fillna("")
gate_summary.loc[gate_summary["gate"].eq("G1"), "status"] = (
    "PASS" if cohort_complete and g1_input_checks["passed"].all() else "FAIL"
)
gate_summary.loc[gate_summary["gate"].eq("G6"), "status"] = (
    "EVIDENCE_COMPLETE_PENDING_REVIEW"
    if len(g6_cohort_checks) and g6_cohort_checks["passed"].all()
    else "FAIL"
)
gate_summary.loc[gate_summary["gate"].eq("G7"), "status"] = (
    "EVIDENCE_COMPLETE_PENDING_REVIEW"
    if len(g7_checks) and g7_checks["passed"].all()
    else "FAIL"
)
gate_summary.loc[gate_summary["gate"].eq("G8"), "status"] = (
    "EVIDENCE_COMPLETE_PENDING_REVIEW"
    if len(g8_checks) and g8_checks["passed"].all()
    else "FAIL"
)
gate_summary.loc[gate_summary["gate"].eq("G9"), "status"] = "N/A"
gate_summary.loc[gate_summary["gate"].eq("G10"), "status"] = "PENDING_SCIENTIFIC_REVIEW"
save_table(gate_summary, VALIDATION / "qadd_v420_gate_summary_cohort")

figure_contract = pd.DataFrame(
    [
        {
            "panel": "A",
            "status": "COMPLETE_PREFLIGHT",
            "purpose": "controlled construct response",
        },
        {
            "panel": "B",
            "status": "COMPLETE_PREFLIGHT",
            "purpose": "discriminant specificity",
        },
        {
            "panel": "C",
            "status": "COMPLETE_PREFLIGHT",
            "purpose": "transformation contract",
        },
        {
            "panel": "D",
            "status": "COMPLETE" if "D_support_and_availability" in completed_cohort_panels else "MISSING",
            "purpose": "support and availability",
        },
        {
            "panel": "E",
            "status": "COMPLETE" if "E_support_boundary_sensitivity" in completed_cohort_panels else "MISSING",
            "purpose": "support and boundary sensitivity",
        },
        {
            "panel": "F",
            "status": "COMPLETE" if "F_empirical_distributions" in completed_cohort_panels else "MISSING",
            "purpose": "empirical distributions",
        },
        {
            "panel": "G",
            "status": "COMPLETE" if gallery_complete else "MISSING_OR_ERROR",
            "purpose": "signal-linked examples",
        },
        {
            "panel": "H",
            "status": "COMPLETE" if "H_reliability_redundancy_weighting" in completed_cohort_panels else "MISSING",
            "purpose": "reliability and redundancy",
        },
        {
            "panel": "I",
            "status": "N/A",
            "purpose": "no retained event detector",
        },
        {
            "panel": "J",
            "status": "COMPLETE" if "J_ml_handoff_contract" in completed_cohort_panels else "MISSING",
            "purpose": "quality-aware ML handoff",
        },
    ]
)
save_table(figure_contract, VALIDATION / "qadd_v420_figure_contract")

candidate_manifest = {
    "measurement_version": MEASUREMENT_VERSION,
    "orchestration_version": COHORT_ORCHESTRATION_VERSION,
    "candidate_only": True,
    "cohort_extraction_completed": cohort_complete,
    "recording_count": int(len(recording_table)),
    "participant_count": int(
        recording_table[subject_column].nunique(dropna=True)
        if len(recording_table)
        else 0
    ),
    "preflight_blocking_checks_pass": bool(
        preflight_manifest.get("preflight_blocking_checks_pass", False)
    ),
    "cohort_evidence_complete": bool(
        cohort_complete
        and len(cohort_checks)
        and cohort_checks["passed"].fillna(False).astype(bool).all()
        and required_cohort_panels.issubset(completed_cohort_panels)
        and gallery_complete
    ),
    "scientific_review_decision": SCIENTIFIC_REVIEW_DECISION,
    "scientific_reviewer": SCIENTIFIC_REVIEWER,
    "scientific_review_rationale": SCIENTIFIC_REVIEW_RATIONALE,
    "freeze_allowed": False,
    "freeze_blocker": "feature-specific G10 scientific review after cohort audit",
    "analysis_features": list(ANALYSIS_FEATURES),
    "family_scalar_constructed": False,
    "standalone_gate_allowed": False,
    "decision_threshold_status": "not_calibrated",
    "canonical_interval_contract": {
        "profile": CANONICAL_PROFILE,
        "primary_view": CANONICAL_PRIMARY_VIEW,
        "speech_view": CANONICAL_SPEECH_VIEW,
        "pause_view": CANONICAL_PAUSE_VIEW,
        "speech_intervals_are_guarded": True,
        "pause_intervals_are_guarded": True,
    },
    "hum_null_calibration": {
        "pool_size": HUM_NULL_POOL_SIZE,
        "iterations": HUM_NULL_ITERATIONS,
        "support_grid": (
            hum_null_grid["window_count"].astype(int).tolist()
            if len(hum_null_grid)
            else []
        ),
        "selection": "exact observed valid-window count; conservative fallback only if an unexpected count is absent",
    },
    "required_panels_complete": bool(
        required_cohort_panels.issubset(completed_cohort_panels)
        and gallery_complete
    ),
    "input_artifact_sha256": {
        row.artifact: row.sha256 for row in input_artifacts.itertuples(index=False)
    },
    "implementation_sha256": sha256_file(
        REVIEWED_SRC / "paper1_qc_reviewed" / "qadd_v420.py"
    ),
    "cohort_orchestration_sha256": sha256_file(
        REVIEWED_SRC / "paper1_qc_reviewed" / "qadd_v420_cohort.py"
    ),
}
write_json(candidate_manifest, MANIFESTS / "qadd_v420_cohort_candidate_manifest.json")
save_table(hash_inventory(STAGE), MANIFESTS / "qadd_v420_candidate_artifact_inventory", parquet=False)

display(gate_summary)
display(figure_contract)
display(feature_decisions)
print("QADD v4.2.0 REVIEWED COHORT RUN COMPLETE")
print(json.dumps(json_safe(candidate_manifest), indent=2))

if PUBLISH_AND_FREEZE:
    raise RuntimeError(
        "This cohort notebook cannot freeze QADD. "
        "Run the post-audit finalization/freeze patch only after G10 decisions."
    )


## Required next action

Save the executed notebook and package the notebook plus the `qadd-v4.2.0-candidate` outputs for scientific review. Do not change `PUBLISH_AND_FREEZE`, and do not manually mark G10 as accepted.

The post-run audit will determine final feature roles, any required revisions, and whether a separate finalization/freeze patch can be issued.
